# 01 データの取得と構造確認（RSNA Pneumonia Detection Challenge 2018）

このノートブックは **Google Colab（自宅）** と **研究室のローカル GPU PC** のどちらでも動きます。

1. 環境の準備（Colab では Google Drive のマウントとリポジトリの取得）
2. 公式 RSNA 配布ファイルのダウンロード（約 4.1 GB、中断しても再実行で続きから再開）
3. 表（`images.csv` など）の作成と DICOM ヘッダの確認
4. データ確認レポートの作成
5. 交差検証の分割（患者単位・5分割）
6. BBox を重ねた画像の確認

この段階では GPU は不要です（Colab は CPU ランタイムで構いません）。
詳しい手順は `docs/setup.md` を参照してください。

## 1. 環境の準備

- **Colab**：Google Drive をマウントし、リポジトリを `/content/graduationResearch` に取得します。データは Drive の `MyDrive/graduationResearch/data` に保存されるので、次回以降はダウンロード不要です。
- **ローカル**：リポジトリのフォルダ（または `notebooks/`）でこのノートブックを開いてください。事前に `pip install -e ".[dev]"` を済ませておきます。

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BRANCH = "claude/ecstatic-sagan-459cpd"  # main にマージした後は "main" に変更
REPO_URL = "https://github.com/ryu93885/graduationResearch.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/graduationResearch")
    if not REPO.exists():
        !git clone -q -b {BRANCH} {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q -e {REPO}
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))  # import without restarting the kernel

# データの保存先を変えたいときは、ここで指定します（例：研究室 PC の別ドライブ）
# os.environ["GR_DATA_DIR"] = r"D:\graduationResearch\data"

from gradresearch import paths

print(paths.describe())

## 2. ダウンロード

- 既定では、アノテーション（JSON 2 種）、NIH との対応表、画像 ZIP（30,000 枚）を取得します。
- 途中で切れても、もう一度このセルを実行すれば続きから再開します。
- ダウンロード後にサイズと SHA-256 を照合し、ZIP の全ファイルの CRC を検査します。結果は `raw/MANIFEST.json` に記録されます。

In [ ]:
!{sys.executable} scripts/download_data.py --list
!{sys.executable} scripts/download_data.py

## 3. 表の作成

ZIP を展開せずに、30,000 枚の DICOM ヘッダを直接読み込みます。2 回目以降はキャッシュ（`processed/dicom_headers.csv`）を使います。

In [ ]:
!{sys.executable} scripts/build_tables.py

In [ ]:
import pandas as pd

images = pd.read_csv(paths.processed_dir() / "images.csv")
boxes = pd.read_csv(paths.processed_dir() / "boxes.csv")
print(images.shape, boxes.shape)
images.head()

## 4. データ確認レポート

ラベルの分布、撮影方向・年齢・性別・PixelSpacing、BBox の統計、読影者の一致度、患者ごとの画像枚数をまとめます。

In [ ]:
!{sys.executable} scripts/inspect_dataset.py

from IPython.display import Markdown, display

display(Markdown((paths.output_dir() / "dataset_inspection" / "report.md").read_text(encoding="utf-8")))

## 5. 交差検証の分割（患者単位・5分割）

同じ患者の画像がすべて同じ組に入るように、5つの組に分けます。最終ラベル・評価用の層（判断が難しい画像など）・撮影方向がそろうように分けます。
表示される指紋（fingerprint）が Colab とローカルで同じなら、同じ分割になっています（2026-11-12 の公式データ・シード 42 では `5f2c2423bfaa75b3`）。

In [ ]:
!{sys.executable} scripts/make_splits.py

display(Markdown((paths.output_dir() / "splits" / "fold_balance.md").read_text(encoding="utf-8")))

## 6. BBox を重ねた画像の確認

ランダムに選んだ陽性画像に、最終ラベルの BBox（赤）を重ねて表示します。`x, y` は左上の座標、`w, h` は幅と高さ（ピクセル）です。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from gradresearch.data.images import read_pixels
from gradresearch.data.sources import SOURCES

zip_path = paths.raw_dir() / SOURCES["images_full"].filename
sample = images[images["target"] == 1].sample(4, random_state=0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, (_, row) in zip(axes, sample.iterrows()):
    ax.imshow(read_pixels(zip_path, row["zip_member"]), cmap="gray")
    for _, b in boxes[boxes["image_id"] == row["image_id"]].iterrows():
        ax.add_patch(Rectangle((b["x"], b["y"]), b["w"], b["h"], fill=False, edgecolor="red", linewidth=1.5))
    ax.set_title(f"{row['view']}  multi_read={row['multi_read']}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. （任意）テストの実行

プログラムの動作確認用です。合成データを使うので、ダウンロードしたデータは使いません。

In [ ]:
!{sys.executable} -m pytest -q